Name : Syed Waleed Hussain - DHC @057 - syedwaleedhussain11@gmail.com

In [8]:
!pip install transformers datasets evaluate accelerate gradio

# Load Dataset

In [16]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Loading Parquet files via Pandas to bypass the URI validation error
train_df = pd.read_parquet("https://huggingface.co/datasets/ag_news/resolve/main/data/train-00000-of-00001.parquet")
test_df = pd.read_parquet("https://huggingface.co/datasets/ag_news/resolve/main/data/test-00000-of-00001.parquet")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})
display(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

# Tokenization

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Ensure dataset is available before mapping
tokenized_datasets = dataset.map(tokenize_function, batched=True)

NameError: name 'dataset' is not defined

# Model Training

In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Use 'eval_strategy' as 'evaluation_strategy' is deprecated
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'].shuffle(seed=42).select(range(100)),
    eval_dataset=tokenized_datasets['test'].shuffle(seed=42).select(range(50)),
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'tokenized_datasets' is not defined

# Deployment with Gradio

In [7]:
!pip install gradio
import gradio as gr

def classify_news(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)
    labels = ['World', 'Sports', 'Business', 'Sci/Tech']
    return {labels[i]: float(probs[0][i]) for i in range(4)}

interface = gr.Interface(fn=classify_news, inputs='text', outputs='label')
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b097770f282e092081.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
